# PrismLab P2 — N3: 10-image OCR benchmark 🧪

**Goal:** Compare **VL-only answer extraction** against **VL transcription → text-model reasoning**, using known-answer synthetic images. We will plot actual latency, answer correctness, and OCR key-phrase recall. **No benchmark measurements are pre-filled.**

**Study design:** Same image and same question for both arms. In the pipeline arm, the LFM3-1.7B model sees the **OCR text only**, not the image. The pipeline therefore does **not** increase OCR accuracy by itself; its value, if any, is in extracting or reasoning from the transcription.

> First run two images, inspect the outputs, then enable the 10-image experiment. These tests are synthetic, not a measure of real-world generalization.

## Cell 1 — Imports and locate dataset folder

In [ ]:
import sys, json, subprocess
from pathlib import Path
import pandas as pd
from IPython.display import display
from PIL import Image

here = Path.cwd().resolve()
candidates = [
    here, here.parent, here.parent.parent,
    here / "prismlab_ocr_suite",
    here.parent / "prismlab_ocr_suite",
]
ROOT = next((p for p in candidates if (p / "manifest.json").exists() and (p / "benchmark.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from inside the extracted prismlab_ocr_suite folder.")
ROOT = ROOT.resolve()
sys.path.insert(0, str(ROOT))
from benchmark import discover_model, evaluate_case, CASES
print("Dataset:", ROOT)
print("Test cases:", len(CASES))

## Cell 2 — Look at the ten test images and their ground-truth questions

In [ ]:
display(Image.open(ROOT / "contact_sheet.png"))

display(pd.DataFrame([{
    "id": c["id"], "category": c["category"],
    "question": c["question"], "expected": c["expected_answer"]
} for c in CASES]))

## Cell 3 — Check the two local llama.cpp servers

Your VL model was on port **8180**; your text model is usually on **8080**. Modify the variables if you used other ports. A running text server is required only for the **both** comparison.

In [ ]:
VL_SERVER = "http://127.0.0.1:8180"
TEXT_SERVER = "http://127.0.0.1:8080"

VL_MODEL = discover_model(VL_SERVER)
print("Vision model:", VL_MODEL)

try:
    TEXT_MODEL = discover_model(TEXT_SERVER)
    print("Text model:", TEXT_MODEL)
    text_ready = True
except Exception as e:
    TEXT_MODEL = None
    text_ready = False
    print("Text server not connected yet:", e)

## Cell 4 — Smoke test: direct vision-answer extraction (2 images)

This uses the local model, not stored fake responses. It saves a CSV and raw JSONL under `results/`.

In [ ]:
cmd = [sys.executable, str(ROOT / "benchmark.py"),
       "--mode", "direct", "--limit", "2", "--tag", "smoke",
       "--vision-url", VL_SERVER]
subprocess.run(cmd, cwd=ROOT, check=True)
display(pd.read_csv(ROOT / "results/smoke_results.csv")[[
    "filename", "expected_answer", "direct_answer", "direct_exact", "direct_latency_s"
]])

## Cell 5 — Pipeline smoke test: 2 images, **VL-only vs VL → Text**

**Both servers must be connected.** This adds the transcription request and then asks LFM3-1.7B to extract the answer from the OCR output.

In [ ]:
if not text_ready:
    print("Start the text server on port 8080, rerun Cell 3, then run this cell again.")
else:
    cmd = [sys.executable, str(ROOT / "benchmark.py"),
           "--mode", "both", "--limit", "2", "--tag", "pilot",
           "--vision-url", VL_SERVER, "--text-url", TEXT_SERVER]
    subprocess.run(cmd, cwd=ROOT, check=True)
    pilot = pd.read_csv(ROOT / "results/pilot_results.csv")
    display(pilot[["filename", "direct_answer", "pipeline_answer",
                    "direct_exact", "pipeline_exact", "ocr_phrase_recall",
                    "direct_latency_s", "pipeline_latency_s"]])

## Cell 6 — Run all 10 images

Change `RUN_FULL` to `True` **after** inspecting the pilot output. This executes up to 30 sequential model calls (10 direct + 10 OCR + 10 text) and records every measurement. Reusing the same tag replaces the previous results, so use a new tag for another pass.

In [ ]:
RUN_FULL = False  # Set to True when ready
TAG = "full_trial1"

if RUN_FULL:
    if not text_ready:
        raise RuntimeError("Text model must be running for both pipelines.")
    cmd = [sys.executable, str(ROOT / "benchmark.py"),
           "--mode", "both", "--limit", "0", "--tag", TAG,
           "--vision-url", VL_SERVER, "--text-url", TEXT_SERVER]
    subprocess.run(cmd, cwd=ROOT, check=True)
    print("Saved:", ROOT / "results" / f"{TAG}_results.csv")
else:
    print("Skipped the full run. Review the pilot, then set RUN_FULL=True.")

## Cell 7 — Plot measured comparisons

Plot the full run if it exists, otherwise the pilot. The code cannot plot data that has not been collected.

In [ ]:
from IPython.display import Image as DisplayImage

full_file = ROOT / "results" / f"{TAG}_results.csv"
pilot_file = ROOT / "results/pilot_results.csv"
smoke_file = ROOT / "results/smoke_results.csv"
results_csv = next((p for p in [full_file, pilot_file, smoke_file] if p.exists()), None)
if results_csv is None:
    raise FileNotFoundError("Run Cell 4 or Cell 5 before plotting.")
subprocess.run([sys.executable, str(ROOT / "plot_results.py"), "--csv", str(results_csv)],
               cwd=ROOT, check=True)
for name in ["latency_comparison.png", "answer_accuracy.png", "ocr_phrase_recall.png"]:
    img = ROOT / "results" / name
    if img.exists():
        display(DisplayImage(filename=str(img)))

## Cell 8 — Inspect incorrect answers and evidence

A failed exact match may be a formatting issue, an OCR miss, or a reasoning failure. Read the raw transcription before making claims about either model.

In [ ]:
df = pd.read_csv(results_csv)
cols = [c for c in ["filename", "question", "expected_answer", "direct_answer",
                   "pipeline_answer", "direct_exact", "pipeline_exact",
                   "ocr_phrase_recall", "ocr_valid", "direct_valid", "pipeline_valid"]
        if c in df.columns]
display(df[cols])

# Inspect the OCR transcription for the first case, if it was collected
if "ocr_text" in df.columns and df["ocr_text"].notna().any():
    print("\n=== First OCR transcription ===\n", df.loc[df["ocr_text"].notna(), "ocr_text"].iloc[0])

## Research checkpoint

1. Does adding the text model improve **exact-answer correctness** on any of these images?
2. Is the added model inference latency justified by the accuracy change?
3. How does OCR key-phrase recall correlate with wrong pipeline answers?
4. Which failures are due to **image reading** versus **reasoning/extraction**?

**Limitations:** synthetic examples; one question per image; no handwritten/real-camera distribution; modest sample; no repeated-trial confidence intervals. This is an informative controlled pilot, not a benchmark paper.
